In [1]:
import numpy as np
import pandas as pd
sms_data = pd.read_csv("spam.csv", encoding='latin-1')
sms_data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [2]:
cols = sms_data.columns[:2]
data = sms_data[cols]
data.shape

(5572, 2)

In [3]:
data = data.rename(columns={"v1":"Value","v2":"Text"})
data.Value.value_counts()

Value
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
from string import punctuation
import re
import nltk
from nltk import word_tokenize
punctuation = list(punctuation)

In [5]:
data["Punctuations"] = data["Text"].apply(lambda x: len(re.findall(r"[^\w+&&^\s]",x)))

C:\Users\Muhamad Asqar\AppData\Local\Temp\ipykernel_19100\3617767462.py:1: FutureWarning: Possible set intersection at position 5
  data["Punctuations"] = data["Text"].apply(lambda x: len(re.findall(r"[^\w+&&^\s]",x)))


In [6]:
data["Phonenumbers"] = data["Text"].apply(lambda x: len(re.findall(r"[0-9]{10}",x)))

In [7]:
is_link = lambda x: 1 if re.search(r"https?://(?:[-\w.]|(?:%[\da-fA-F]{2}))+",x)!=None else 0
data["Links"] = data["Text"].apply(is_link)

In [8]:
count_upper = lambda x : list(map(str.isupper,x.split())).count(True) 
upper_case = lambda y,n : n+1 if y.isupper() else n
data["Uppercase"] = data["Text"].apply(count_upper)

In [9]:
def find_unusual_words(text):
    text_vocab_set = set(w.lower() for w in text if w.isalpha())
    english_vocab_set = set(w.lower() for w in nltk.corpus.words.words())
    unusual_set = text_vocab_set - english_vocab_set
    return len(sorted(unusual_set))
data["unusualwords"] = data["Text"].apply(lambda x: find_unusual_words(word_tokenize(x)))

In [10]:
data[14:25]

,Value,Text,Punctuations,Phonenumbers,Links,Uppercase,unusualwords
14,ham,I HAVE A DATE ON SUNDAY WITH WILL!!,2,0,0,8,0
15,spam,"XXXMobileMovieClub: To use your credit, click ...",11,0,1,1,3
16,ham,Oh k...i'm watching here:),6,0,0,0,0
17,ham,Eh u remember how 2 spell his name... Yes i di...,5,0,0,0,0
18,ham,Fine if thatåÕs the way u feel. ThatåÕs the wa...,1,0,0,0,2
19,spam,England v Macedonia - dont miss the goals/team...,7,0,0,2,6
20,ham,Is that seriously how you spell his name?,1,0,0,0,0
21,ham,IÛ÷m going to try for 2 months ha ha only joking,2,0,0,0,2
22,ham,So Ì_ pay first lar... Then when is da stock c...,6,0,0,1,1
23,ham,Aft i finish my lunch then i go str down lor. ...,3,0,0,1,4


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf= TfidfVectorizer(stop_words="english",strip_accents='ascii',max_features=300)
tf_idf_matrix = tf_idf.fit_transform(data["Text"])

In [12]:
data_extra_features = pd.concat([data,pd.DataFrame(tf_idf_matrix.toarray(),columns=tf_idf.get_feature_names_out())],axis=1)

In [13]:
from sklearn.model_selection import train_test_split
X = data_extra_features
features = X.columns.drop(["Value","Text"])
target = ["Value"]
X_train,X_test,y_train,y_test = train_test_split(X[features],X[target])

In [14]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
dt = DecisionTreeClassifier(min_samples_split=40)
dt.fit(X_train,y_train)
pred = dt.predict(X_test)
print(f"Accuracy (Training): {accuracy_score(y_train, dt.predict(X_train)) * 100 :.2f}%")
print(f"Accuracy (Testing): {accuracy_score(y_test, pred) * 100 :.2f}%")

Accuracy (Training): 98.64%
Accuracy (Testing): 97.34%


In [15]:
from sklearn.naive_bayes import MultinomialNB
mnb = MultinomialNB()
mnb.fit(X_train,y_train)
pred_mnb = mnb.predict(X_test)
print(f"Accuracy (Training): {accuracy_score(y_train, mnb.predict(X_train)) * 100 :.2f}%")
print(f"Accuracy (Testing): {accuracy_score(y_test, pred_mnb) * 100 :.2f}%")

Accuracy (Training): 97.03%
Accuracy (Testing): 96.55%


c:\Users\Muhamad Asqar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [16]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train,y_train)
pred_lr = lr.predict(X_test)
print(f"Accuracy (Training): {accuracy_score(y_train, lr.predict(X_train)) * 100 :.2f}%")
print(f"Accuracy (Testing): {accuracy_score(y_test, pred_lr) * 100 :.2f}%")

Accuracy (Training): 98.23%
Accuracy (Testing): 97.56%


c:\Users\Muhamad Asqar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [17]:
import joblib
joblib.dump(tf_idf, "spam_vectorizer.joblib")
joblib.dump(dt,"spam_model.joblib")

['spam_model.joblib']